<a href="https://colab.research.google.com/github/cbonnin88/MapleFit/blob/main/MapleFit_dataGeneration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import polars as pl
import numpy as np
from datetime import datetime, timedelta

In [2]:
np.random.seed(42)
num_users = 5000

In [3]:
cities_map = {
    'Canada': {'names':["Toronto", "Vancouver", "Montreal", "Calgary", "Ottawa"],
               "weights": [0.4, 0.2, 0.2, 0.1, 0.1]},
    "United Kingdom": {"names": ["London", "Manchester", "Birmingham", "Glasgow", "Liverpool"],
                       "weights": [0.5, 0.15, 0.15, 0.1, 0.1]}
}

In [4]:
# Base User Data
countries = np.random.choice(['Canada','United Kingdom'], size=num_users,p=[0.4,0.6])
city_list = [np.random.choice(cities_map[c]['names'],p=cities_map[c]['weights']) for c in countries]

In [5]:
# Tier and Plan Logic
tiers = ['Free','Basic','Premium']
tier_choices = np.random.choice(tiers, size=num_users, p=[0.6,0.25,0.15])

In [6]:
# Map tiers to CAD prices
plan_map = {'Free': 0.00,'Basic':4.99,'Premium': 10.99}
prices = [plan_map[t] for t in tier_choices]

In [7]:
users = pl.DataFrame({
    'user_id': np.arange(1,num_users +1),
    'signup_date': [datetime(2025,1,1) + timedelta(days=np.random.randint(0,365)) for _ in range(num_users)],
    'country': countries,
    'city': city_list,
    'membership_tier': tier_choices,
    'membership_plan_cad': prices,
    'age': np.random.randint(18,65, size=num_users)
})

In [8]:
display(users.head())

user_id,signup_date,country,city,membership_tier,membership_plan_cad,age
i64,datetime[μs],str,str,str,f64,i64
1,2025-10-05 00:00:00,"""Canada""","""Toronto""","""Free""",0.0,21
2,2025-01-16 00:00:00,"""United Kingdom""","""London""","""Free""",0.0,37
3,2025-01-13 00:00:00,"""United Kingdom""","""Glasgow""","""Free""",0.0,28
4,2025-12-22 00:00:00,"""United Kingdom""","""London""","""Basic""",4.99,59
5,2025-12-22 00:00:00,"""Canada""","""Calgary""","""Free""",0.0,39


In [9]:
event_types = [
    "session_start",
    "workout_start",
    "workout_completed",
    "meal_logged",
    "social_share",
    "profile_update",
    "subscription_renewed",
    "app_crash"
]

event_weights = [0.35, 0.20, 0.15, 0.12, 0.05, 0.03, 0.05, 0.05]

In [13]:
event_data = []
for row in users.iter_rows(named=True):
  n_events = np.random.randint(2,60) if row['membership_tier'] != 'Free' else np.random.randint(1,15)

  for _ in range(n_events):
    days_after = np.random.randint(0,60) # Activities over 2 months
    ts = row['signup_date'] + timedelta(days=days_after, minutes=np.random.randint(0,1440))

    event_data.append({
        'user_id': row['user_id'],
        'event_type': np.random.choice(event_types, p=event_weights),
        'timestamp': ts
    })

events = pl.DataFrame(event_data)

# **Data Cleaning**

In [14]:
# 1. Deduplication
events = events.unique()

In [15]:
# 2. Sort and enforce types
events = events.with_columns(
    pl.col('timestamp').cast(pl.Datetime)
).sort('timestamp')

In [16]:
# 3. Currency Normalization
# Even though membership is in CAD, I flag UK users for a later FX conversion in dbt
users = users.with_columns(
    preferred_currency = pl.when(pl.col('country') == 'United Kingdom').then(pl.lit("GBP")).otherwise(pl.lit('CAD'))
)

In [17]:
# 4. Fitler out technical errors for a 'Clean' event view (optional but good practice)
clean_events = events.filter(pl.col('event_type') != 'app_crash')

In [18]:
# Export for BigQuery
users.write_csv('maplefit_users.csv')
events.write_csv('maplefit_events.csv')

print(f'Data Generation Complete !')
print(f'Total Events: {len(events)} | Percentage of app crashes: {(events.filter(pl.col('event_type')=='app_crash').height / len(events)):.2%}')

Data Generation Complete !
Total Events: 83877 | Percentage of app crashes: 5.11%
